In [ ]:
# ==========================================================
# IMPORTAÇÃO DAS BIBLIOTECAS
# ==========================================================

# pandas -> leitura e manipulação do dataset
import pandas as pd

# math -> utilizar a raiz quadrada
import math

# numpy -> Cálculos Matemáticos
import numpy as np

# Counter -> contar qual classe aparece mais vezes
from collections import Counter

# Métricas para avaliação do modelo
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [ ]:
# ==========================================================
# LEITURA DO DATASET
# ==========================================================

# Carrega o arquivo CSV
df = pd.read_csv("https://drive.google.com/uc?export=download&id=1wF7amDwgNKIxa9Lc5Dm4OXkiDy_N_CYu")

# Embaralha os dados
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Mostra as 5 primeiras linhas
print(df.head())

# Mostra quantidade de linhas e colunas
print("\nFormato do dataset:", df.shape)


   battery_power  bluetooth  clock_speed  dual_sim  front_camera  four_g  \
0           1392          0          2.5         0             0       1   
1           1503          1          0.7         0            10       1   
2            765          0          2.9         0             0       1   
3            754          0          0.5         1             7       1   
4           1811          1          2.5         0             4       1   

   storage  mobile_weight  n_cores  primary_camera  px_height  px_width   ram  \
0       44            113        4               2        482      1098  1280   
1       43            166        4              14        841      1304  2122   
2       18            153        7               0          6       793  1066   
3       59            178        7              10       1914      1928  1027   
4        5             98        4              16        447       568  2700   

   sc_height  sc_width  talk_time  three_g  touch_screen

In [ ]:
# ==========================================================
# SEPARAÇÃO DOS ATRIBUTOS E DA VARIÁVEL ALVO
# ==========================================================

# Lista com as 8 colunas escolhidas para o treino
colunas_usadas = [
    "battery_power",
    "clock_speed",
    "storage",
    "n_cores",
    "primary_camera",
    "px_width",
    "ram",
    "talk_time"
]

# X -> atributos selecionados para o treinamento
X = df[colunas_usadas].values

# y -> variável alvo
y = df["price_range"].values

In [ ]:
# ==========================================================
# NORMALIZAÇÃO MANUAL DOS DADOS
# ==========================================================

# Converte os atributos para valores numéricos do tipo float
X = np.array(X, dtype=float)

# Calcula mínimos e máximos de cada coluna
minimos = X.min(axis=0)
maximos = X.max(axis=0)

# Calcula o denominador da fórmula de normalização
denominador = maximos - minimos

# Evita divisão por zero caso alguma coluna tenha todos os valores iguais
denominador[denominador == 0] = 1

# Aplica a normalização Min-Max
X = (X - minimos) / denominador

print("\nDados normalizados com sucesso!")


Dados normalizados com sucesso!


In [ ]:
# ==========================================================
# DIVISÃO ENTRE TREINO E TESTE
# ==========================================================

# Utiliza 80% para treino
tamanho_treino = int(0.8 * len(X))

# Dados de treino
X_treino = X[:tamanho_treino]
y_treino = y[:tamanho_treino]

# Dados de teste
X_teste = X[tamanho_treino:]
y_teste = y[tamanho_treino:]

print("\nQuantidade treino:", len(X_treino))
print("Quantidade teste:", len(X_teste))


Quantidade treino: 1598
Quantidade teste: 400


In [ ]:
# ==========================================================
# FUNÇÃO DA DISTÂNCIA EUCLIDIANA
# ==========================================================

# Calcula a distância entre dois pontos
def distancia_euclidiana(ponto1, ponto2):

    diferenca = ponto1 - ponto2

    distancia = np.sqrt(np.sum(diferenca ** 2))

    return distancia

In [ ]:
# ==========================================================
# FUNÇÃO PRINCIPAL DO KNN
# ==========================================================

def knn_prever(X_treino, y_treino, novo_ponto, k):

    # Lista para armazenar distâncias
    distancias = []

    # ======================================================
    # CALCULAR DISTÂNCIA ENTRE O NOVO PONTO
    # E TODOS OS DADOS DE TREINO
    # ======================================================

    for i in range(len(X_treino)):

        distancia = distancia_euclidiana(
            novo_ponto,
            X_treino[i]
        )

        classe = y_treino[i]

        # Armazena distância e classe
        distancias.append((distancia, classe))


    # ======================================================
    # ORDENAR DA MENOR DISTÂNCIA PARA A MAIOR
    # ======================================================

    distancias.sort(key=lambda item: item[0])


    # ======================================================
    # PEGAR OS K VIZINHOS MAIS PRÓXIMOS
    # ======================================================

    vizinhos = distancias[:k]


    # ======================================================
    # PEGAR AS CLASSES DOS VIZINHOS
    # ======================================================

    classes_vizinhos = []

    for vizinho in vizinhos:
        classes_vizinhos.append(vizinho[1])


    # ======================================================
    # DESCOBRIR QUAL CLASSE APARECE MAIS
    # ======================================================

    classe_prevista = Counter(classes_vizinhos).most_common(1)[0][0]

    return classe_prevista

In [ ]:
# ==========================================================
# ESCOLHA DO VALOR DE K
# ==========================================================

# Quantidade de vizinhos
k = 31

print("\nValor escolhido para K:", k)


Valor escolhido para K: 31


In [ ]:
# ==========================================================
# REALIZAR PREVISÕES
# ==========================================================

previsoes = []

# Percorre todos os dados de teste
for ponto in X_teste:

    # Faz previsão
    classe_prevista = knn_prever(
        X_treino,
        y_treino,
        ponto,
        k
    )

    # Salva previsão
    previsoes.append(classe_prevista)

In [ ]:
# ==========================================================
# CALCULAR ACURÁCIA
# ==========================================================

acertos = 0

# Verifica quantas previsões acertaram
for i in range(len(y_teste)):

    if previsoes[i] == y_teste[i]:
        acertos += 1

# Fórmula da acurácia
acuracia = acertos / len(y_teste)

In [ ]:
# ==========================================================
# RESULTADOS FINAIS
# ==========================================================

print("\n================ RESULTADOS ================")

print("Total de testes:", len(y_teste))

print("Total de acertos:", acertos)

print("Acurácia:", acuracia)

print("Acurácia em porcentagem:",
      round(acuracia * 100, 2), "%")


================ RESULTADOS ================
Total de testes: 400
Total de acertos: 284
Acurácia: 0.71
Acurácia em porcentagem: 71.0 %


In [ ]:
# ==========================================================
# CLASSIFICATION REPORT
# ==========================================================

print("\n================ CLASSIFICATION REPORT ================")

print(classification_report(y_teste, previsoes))


# ==========================================================
# MATRIZ DE CONFUSÃO
# ==========================================================

print("\n================ MATRIZ DE CONFUSÃO ================")

print(confusion_matrix(y_teste, previsoes))


================ CLASSIFICATION REPORT ================
              precision    recall  f1-score   support

           0       0.82      0.82      0.82        98
           1       0.63      0.55      0.59       110
           2       0.52      0.73      0.61        86
           3       0.94      0.76      0.84       106

    accuracy                           0.71       400
   macro avg       0.73      0.71      0.71       400
weighted avg       0.74      0.71      0.72       400


================ MATRIZ DE CONFUSÃO ================
[[80 17  1  0]
 [17 60 33  0]
 [ 1 17 63  5]
 [ 0  1 24 81]]


### Usando todas as colunas para treino!   


### RESULTADOS COM K=1
Acurácia em porcentagem: 39 % / f1-score 0.39

### RESULTADOS COM K=9
Acurácia em porcentagem: 45.75 % / f1-score 0.46    

### RESULTADOS COM K=15
Acurácia em porcentagem: 46.25 % / f1-score 0.47   

### RESULTADOS COM K=21
Acurácia em porcentagem: 48.50 % / f1-score 0.49   

### RESULTADOS COM K=39 *********
Acurácia em porcentagem: 57.25 % / f1-score 0.58

### RESULTADOS COM K=41
Acurácia em porcentagem: 56.25 % / f1-score 0.56

### RESULTADOS COM K=45
Acurácia em porcentagem: 53 % / f1-score 0.53





### Usando 14 colunas para treino!   


### RESULTADOS COM K=1
Acurácia em porcentagem: 51.5 % / f1-score 0.52

### RESULTADOS COM K=9
Acurácia em porcentagem: 53.25 % / f1-score 0.54    

### RESULTADOS COM K=15
Acurácia em porcentagem: 54.50 % / f1-score 0.55   

### RESULTADOS COM K=21
Acurácia em porcentagem: 59.75 % / f1-score 0.60   

### RESULTADOS COM K=39
Acurácia em porcentagem: 62.25 % / f1-score 0.62

### RESULTADOS COM K=41 ****************
Acurácia em porcentagem: 63.75 % / f1-score 0.64

### RESULTADOS COM K=45
Acurácia em porcentagem: 61.25 % / f1-score 0.61





### Usando 8 colunas para treino!   


### RESULTADOS COM K=1
Acurácia em porcentagem: 61.25 % / f1-score 0.61

### RESULTADOS COM K=9
Acurácia em porcentagem: 69.50 % / f1-score 0.70    

### RESULTADOS COM K=15***********
Acurácia em porcentagem: 73.50 % / f1-score 0.74   

### RESULTADOS COM K=21
Acurácia em porcentagem: 72.50 % / f1-score 0.72   

### RESULTADOS COM K=25
Acurácia em porcentagem: 72 % / f1-score 0.72

### RESULTADOS COM K=31
Acurácia em porcentagem: 71 % / f1-score 0.71







Colunas Usadas no treino de 14:

    battery_power",
    "clock_speed",
    "storage",
    "mobile_weight",
    "n_cores",
    "primary_camera",
    "px_height",
    "px_width",
    "ram",
    "sc_height",
    "sc_width",
    "talk_time",
    "four_g",
    "wifi"

Colunas Usadas no treino de 8:

    "battery_power",
    "clock_speed",
    "storage",
    "n_cores",
    "primary_camera",
    "px_width",
    "ram",
    "talk_time"

